# Homework — Grouping, aggregating & merging data

So far you've *made*, *filtered*, and *sorted* DataFrames — one row at a time.
This homework is about the next big idea: **summarizing many rows into a few**
(grouping) and **combining two tables into one** (merging). These are the moves
you'll use constantly on real lab data.

**How this works**
1. Run the **setup** cell first — it installs pandas and builds the data table.
2. Read each short teaching section and run its example.
3. Answer the numbered **Problems** in the empty cells. About half of them ask you
   to *predict, interpret, or explain* before you code — that's the point.
4. At the very end there's a **check-in value** — type it back into the CHEM 438
   app to mark this homework done.

Run every cell top to bottom. If you skip the setup cell, nothing else will work.

In [ ]:
# Setup — run this first
!pip install pandas -q
import pandas as pd

# One small table of lab results we'll use all the way through.
# Each row is one reaction: which lab ran it, which reagent, and the % yield.
samples = pd.DataFrame({
    "lab":       ["North", "North", "North", "South", "South", "South", "East", "East"],
    "reagent":   ["acid",  "base",  "acid",  "acid",  "base",  "acid",  "acid", "base"],
    "yield_pct": [45,      30,      40,      12,      20,      18,      50,     15],
})
print(samples)

## 1. `groupby` — and why it always needs an aggregation

Say you want the **average yield for each lab**. There are 8 rows but only 3 labs,
so the answer should have **3 numbers**. `groupby` does this in two steps:

1. **Split** the rows into groups that share a value (here, the same `lab`).
2. **Aggregate** — collapse each group down to *one* summary number
   with something like `.mean()`, `.sum()`, `.size()`, or `.agg(...)`.

The key idea: **`groupby` by itself doesn't return a table.** It just holds the
groups, waiting. You *must* tell it how to summarize each group. That summary step
is why the result has **one row per group** instead of one row per original sample.

In [ ]:
# Step 1 on its own: groupby with no aggregation is a "lazy" object, not a table.
print(samples.groupby("lab"))
# ...<pandas ... DataFrameGroupBy object at 0x...>   <- not the answer yet!

# Step 2: add an aggregation. Now we get one row per lab.
print(samples.groupby("lab")["yield_pct"].mean())
# lab
# East     32.500000
# North    38.333333
# South    16.666667
# Name: yield_pct, dtype: float64

**Problem 1 (predict, then run).**
Before running anything, write down your answer to each as a `# comment`:

- (a) `samples` has 8 rows and 3 labs. How many **rows** will
  `samples.groupby("lab")["yield_pct"].sum()` produce?
- (b) In one sentence, *why* does grouping by `lab` give that number of rows
  instead of 8?

Then run the `.sum()` and check whether your (a) was right.

## 2. Grouping by two columns

You can split on **more than one column** by passing a list. Now a "group" is every
unique *combination* that actually appears in the data — e.g. `North + acid`,
`North + base`, and so on. The aggregation still collapses each group to one row.

In [ ]:
print(samples.groupby(["lab", "reagent"])["yield_pct"].mean())
# lab    reagent
# East   acid       50.0
#        base       15.0
# North  acid       42.5
#        base       30.0
# South  acid       15.0
#        base       20.0
# Name: yield_pct, dtype: float64

**Problem 2 (predict, then run).**
There are 3 labs and 2 reagents. A beginner guesses the two-column group above will
have `3 x 2 = 6` rows.

- (a) Write your predicted row count as a `# comment`, and say whether "labs times
  reagents" is *always* the right way to count group rows.
- (b) Run the grouping and confirm the count. (Hint: it happens to be 6 here — but
  would it still be 6 if the `East` lab had *never* used `base`? Answer in a comment.)

## 3. Counting categories with `.value_counts()`

Often you don't want an average — you just want **"how many of each?"** That's
`.value_counts()`. It works on a single column and returns each distinct value with
its count, sorted from most to least common. Think of it as a fast tally.

In [ ]:
print(samples["reagent"].value_counts())
# reagent
# acid    5
# base    3
# Name: count, dtype: int64

**Problem 3 (interpret).**
First, run `samples["lab"].value_counts()` to see how many reactions each lab ran.

Then look back at the per-lab averages from Section 1:
**North averages 38.3% yield, East 32.5%, South 16.7%.**
Your PI asks: *"Which lab needs help, and is a low count enough to explain it?"*
Write a 2-3 sentence answer as `# comments` — use **both** the averages and the
counts. (Is South low because it's unlucky with few runs, or is something else off?)

**Problem 4 (concept check — pick one).**
You run `samples.groupby("lab")["yield_pct"].mean()` and get **3 rows**, even though
`samples` has 8 rows. Which statement best explains *why*?

- **A.** `.mean()` deletes rows at random until 3 are left.
- **B.** `groupby` splits the 8 rows into groups that share a `lab`, and `.mean()`
  collapses each group into a single summary number — so you get one row per lab.
- **C.** pandas always returns 3 rows from any `groupby`.
- **D.** The original table secretly only had 3 rows.

In the empty cell, set `answer = "B"` (or whichever you pick) and print it.

## 5. Merging two tables: inner vs. left join

Real data is spread across tables. `pd.merge` lines up two DataFrames on a shared
**key column** and stitches matching rows together. The big decision is *what to do
with rows that have no match on the other side*:

- **inner join** (`how="inner"`, the default): keep **only** keys that appear in
  **both** tables. Unmatched rows are **dropped**.
- **left join** (`how="left"`): keep **every** row from the left table. Where the
  right table has no match, the new columns are filled with `NaN` (missing).

Below, `totals` has labs North/South/**East**, and `supervisors` has labs
North/South/**West**. Watch what happens to East and West.

In [ ]:
totals = pd.DataFrame({"lab": ["North", "South", "East"],  "n_runs": [3, 3, 2]})
supervisors = pd.DataFrame({"lab": ["North", "South", "West"], "supervisor": ["Ada", "Ben", "Dan"]})

print("INNER (only labs in BOTH tables):")
print(pd.merge(totals, supervisors, on="lab"))
#      lab  n_runs supervisor
# 0  North       3        Ada
# 1  South       3        Ben      <- East AND West both vanished

print("\nLEFT (every row from totals kept):")
print(pd.merge(totals, supervisors, on="lab", how="left"))
#      lab  n_runs supervisor
# 0  North       3        Ada
# 1  South       3        Ben
# 2   East       2        NaN      <- East stayed, supervisor is missing

**Problem 5 (compare the two joins).**
Using the exact output above, answer as `# comments`:

- (a) The **inner** join returned 2 rows. Name the row(s) that disappeared **from
  each** table and say *why* each one was dropped.
- (b) The **left** join returned 3 rows. Why did `West` still *not* appear, even
  though we kept "every row"? (Hint: which table is the *left* one?)
- (c) You need a report listing *all three* labs' run counts, supervisor or not.
  Which join do you use — and what would you see in the `supervisor` column for East?

**Problem 6 (read the error, then fix it).**
Copy the buggy line below into the empty cell and run it. It **crashes**. Read the
error message, then fix the *one thing* that's wrong so it does an inner merge.

```python
pd.merge(totals, supervisors, on="Lab")
```

In a `# comment`, write (a) what the error type/message was, and (b) the one-word
change that fixed it. (Lesson: `merge` keys must match a real column name *exactly*,
capitalization included.)

## 7. A quick `pivot_table`

A **pivot table** is a two-column group reshaped into a grid: one variable down the
rows, another across the columns, and an aggregated value in each cell. It's the same
split-and-aggregate idea as `groupby`, just laid out as a table you can read at a
glance.

In [ ]:
print(pd.pivot_table(samples, index="lab", columns="reagent",
                     values="yield_pct", aggfunc="mean"))
# reagent  acid  base
# lab
# East     50.0  15.0
# North    42.5  30.0
# South    15.0  20.0

**Problem 7.**
Build a `pivot_table` from `samples` with `lab` down the rows and `reagent` across
the columns, but this time use `aggfunc="size"` (a count) instead of `"mean"`.

Then, in a `# comment`: several cells of your grid will read **1**, meaning that
lab+reagent combo was run only once. Name one such cell, and explain why the *average*
yield reported for a count-of-1 cell (in the earlier mean pivot) is fragile / not very
trustworthy compared to a cell backed by more reactions.

## Check in

Run the cell below. It groups the samples by lab, takes each lab's **average yield**,
and prints the **highest** one (rounded to 1 decimal). That single number is your
check-in value.

**Type that number into the CHEM 438 app to mark this homework complete.**

In [ ]:
best = samples.groupby("lab")["yield_pct"].mean().max()
print("Your check-in value:", round(best, 1))